# Research 48 — GOSS and one-hot fixed blend screen

Compare each candidate in its predeclared fixed 80/20 reference blend. The GOSS candidate uses CV seed 42; the one-hot repeat uses CV seed 2025. Include the existing four-seed leader as context and report corresponding fold-slice deltas. No fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert (sums > 0).all(), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
p_leader = load_probs("segment-vip-region-four-seed-average-oof.csv")
p_goss = load_probs("segment-lightgbm-goss-vip-region-oof.csv")
p_onehot = load_probs("segment-lightgbm-onehot-vip-region-seed-2025-oof.csv")
p_goss_blend = 0.8 * p_base + 0.2 * p_goss
p_onehot_blend = 0.8 * p_base + 0.2 * p_onehot
folds = {
    seed: list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    )
    for seed in [42, 2025]
}
candidates = {
    "Current reference": (p_base, None),
    "Four-seed VIP × region leader": (p_leader, None),
    "GOSS standalone, seed 42": (p_goss, 42),
    "GOSS fixed 80/20 blend, seed 42": (p_goss_blend, 42),
    "One-hot standalone, seed 2025": (p_onehot, 2025),
    "One-hot fixed 80/20 blend, seed 2025": (p_onehot_blend, 2025),
}
rows = []
for name, (probs, fold_seed) in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    if fold_seed is not None:
        for fold, (_, valid_idx) in enumerate(folds[fold_seed], start=1):
            row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
            base_fold = f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro")
            row[f"fold_{fold}_delta_vs_reference"] = row[f"fold_{fold}_macro_f1"] - base_fold
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-goss-onehot-blend-screen.csv", index=False)
for _name, probs, filename in [
    ("GOSS", p_goss_blend, "segment-lightgbm-goss-vip-region-blend-oof.csv"),
    (
        "one-hot seed 2025",
        p_onehot_blend,
        "segment-lightgbm-onehot-vip-region-seed-2025-blend-oof.csv",
    ),
]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / "predictions" / filename, index=False
    )
print("Fixed candidate-only 80/20 blends evaluated; no weights searched; Test not loaded.")

,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_1_delta_vs_reference,fold_2_macro_f1,fold_2_delta_vs_reference,fold_3_macro_f1,fold_3_delta_vs_reference
0,Current reference,0.750895,0.764683,NaN,NaN,NaN,NaN,NaN,NaN
1,Four-seed VIP × region leader,0.752172,0.765704,NaN,NaN,NaN,NaN,NaN,NaN
2,"GOSS standalone, seed 42",0.741463,0.752899,0.739961,-0.009485,0.740497,-0.009324,0.743913,-0.009477
3,"GOSS fixed 80/20 blend, seed 42",0.751425,0.764991,0.749436,-0.000010,0.750123,0.000302,0.754690,0.001300
4,"One-hot standalone, seed 2025",0.744120,0.755636,0.744023,-0.006774,0.742345,-0.007067,0.745976,-0.006486
5,"One-hot fixed 80/20 blend, seed 2025",0.751457,0.764999,0.751245,0.000448,0.750191,0.000779,0.752927,0.000465


Fixed candidate-only 80/20 blends evaluated; no weights searched; Test not loaded.
